# ALOHA 验证集预测查看

读取 `train_state_policy.py` 已生成的 `predictions.npz`、`samples.json` 和 `metrics.json`。
选择验证样本，展示真实采集画面、预测 action 与真实 action 曲线。**画面不是模型执行结果；当前 state-only 模型训练时也没有使用这些图像。**

只需 CPU，不需要 PyTorch、额外端口或重新训练。依赖 numpy、ipywidgets、Daft 和 LakeSoul；缺少 ipywidgets 时可运行 `%pip install ipywidgets` 后重启内核。
三个输入文件应来自同一次训练。图片从样本引用的 Prepared 回读，文件路径必须在当前 Notebook 内核可见。

相机按实际表结构自动识别，以两列排列；四路表显示四路，旧两路表仍可使用。这里只支持逐帧 JPEG 图片列，不直接解码 GOP 表。


## 第 1 步：配置训练输出目录

训练未传 `--output` 时，直接使用默认 `OUTPUT_DIR`：同目录 `common.py` 旁的 `state_policy_output/`。训练和查看器必须使用同一份 `common.py`、且输出目录对内核可见。训练显式传了 `--output` 时，将下面 `OUTPUT_DIR` 改为对应的 `Path("实际路径")`。这里不训练，也不创建样本表。


In [ ]:
import html
import json
import os
from pathlib import Path

import numpy as np
import ipywidgets as widgets
from IPython.display import display
from daft import col
import pyarrow.dataset as ds
from lakesoul.catalog import LakeSoulCatalog

from common import DEFAULT_STATE_POLICY_OUTPUT

# 默认与训练脚本一致；训练显式传了 --output 时再改为那个路径。
OUTPUT_DIR = DEFAULT_STATE_POLICY_OUTPUT
print("读取训练输出目录：", OUTPUT_DIR.resolve())


# 根据表中实际存在的相机列读取；兼容旧两路表和新四路表。
CAMERA_NAMES = ("cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist")


def make_camera_grid(width=360):
    images = {name: widgets.Image(format="jpeg", width=width) for name in CAMERA_NAMES}
    panels = {name: widgets.VBox([widgets.HTML(f"<b>{name}</b>"), images[name]],
                                layout=widgets.Layout(display="none"))
              for name in CAMERA_NAMES}
    grid = widgets.GridBox(list(panels.values()), layout=widgets.Layout(
        grid_template_columns=f"repeat(2, {width}px)", grid_gap="8px"))
    return images, panels, grid


def update_camera_images(images, panels, row=None):
    # 无此列则隐藏；清空或切换样本时同时清掉旧图片。
    row = {} if row is None else row
    for name, image in images.items():
        image.value = bytes(row.get(name) or b"")
        panels[name].layout.display = "" if name in row else "none"


## 第 2 步：读取预测和指标

数组第 i 个样本对应 `samples.json` 的 `val[i]`，保持保存时的顺序，不能重新排序。
预测和真实 action 都已恢复为原始数值尺度；各维度含义和单位以源数据定义为准，不自行命名为某个关节。

读取训练时已保存的预测、真实动作和验证样本索引。`samples.json` 是训练实际选中的样本表记录，不是一份需要你重新制作的数据。不能直接拿当前样本表的所有行来配预测：数量和顺序可能不同。


In [ ]:
# 预测文件已经保存了模型输出，不需要加载 PyTorch 或重新推理。
with np.load(OUTPUT_DIR / "predictions.npz", allow_pickle=False) as saved:
    predicted = saved["predicted_action"].copy()
    targets = saved["target_action"].copy()
    states = saved["state"].copy()
provenance = json.loads((OUTPUT_DIR / "samples.json").read_text())
# 保留训练保存时的顺序：第 i 条索引对应第 i 份预测。
indices = provenance["val"]
metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text())
if predicted.shape != targets.shape or predicted.ndim != 3 or len(indices) != len(predicted) or len(states) != len(indices):
    raise ValueError("预测数组和验证样本索引不匹配，请使用同一次训练的输出")
if not indices or any(item["split"] != "val" for item in indices):
    raise ValueError("没有验证样本，或索引包含非 val 样本")
if not all(np.isfinite(a).all() for a in (predicted, targets, states)):
    raise ValueError("预测或目标包含非有限数")
print("验证样本数：", len(indices), "；预测数组形状：", predicted.shape)
print("最佳归一化验证 MSE：", metrics["best_val_mse_normalized"])
print("平均动作基线 MSE：", metrics["mean_action_baseline_val_mse_normalized"])
print("下面的曲线使用原始 action 数值，不是归一化误差。")
if states.ndim != 3 or min(predicted.shape) < 1 or min(states.shape) < 1:
    raise ValueError("state/action 必须是非空的 [样本, 时间步, 维度] 数组")
for index in indices:
    for name, size in (("observation", states.shape[1]), ("action", targets.shape[1])):
        if index[f"{name}_end_exclusive"] - index[f"{name}_start"] != size:
            raise ValueError("样本索引的窗口长度与预测文件不一致")


## 第 3 步：连接 LakeSoul，定义读取与曲线函数
只在加载样本时读取其窗口，不加载整个训练集的图片。回读 state/action 与保存的验证数据核对，防止 Prepared 改变后展示错配。

连接已有 LakeSoul，只回读选中样本窗口的画面。两个辅助函数分别负责“取画面并核对对应关系”和“画真实/预测动作曲线”；不处理或重写 Prepared。核对 state/action 是为了发现表被修改造成的错配，但不等同于固定数据快照。


In [ ]:
options = {"fs.s3a.path.style.access": "true"}
endpoint = os.getenv("AWS_ENDPOINT") or os.getenv("AWS_ENDPOINT_URL")
if endpoint:
    options["fs.s3a.endpoint"] = endpoint
for env, key in [("AWS_ACCESS_KEY_ID", "fs.s3a.access.key"), ("AWS_SECRET_ACCESS_KEY", "fs.s3a.secret.key")]:
    if os.getenv(env):
        options[key] = os.environ[env]
catalog = LakeSoulCatalog.from_env(namespace=indices[0]["prepared_namespace"], object_store_options=options)


# 从已有 Prepared 取当前样本的观察/动作窗口，不扫描全部图片到内存。
def read_window(i):
    index = indices[i]
    start = min(index["observation_start"], index["action_start"])
    end = max(index["observation_end_exclusive"], index["action_end_exclusive"])
    columns = ["episode_id", "frame_index", "timestamp", "split", "valid_interval_id",
               "observation_state", "action"]
    table = catalog.table(index["prepared_table"], namespace=index["prepared_namespace"])
    columns += [name for name in CAMERA_NAMES if name in table.schema.names]
    rows = table.scan(columns=columns, filter=(
        (ds.field("episode_id") == index["episode_id"])
        & (ds.field("frame_index") >= start) & (ds.field("frame_index") < end)
    )).to_daft().where(
        (col("episode_id") == index["episode_id"])
        & (col("split") == index["split"])
        & (col("valid_interval_id") == index["valid_interval_id"])
    ).to_arrow().to_pylist()
    # 用原始帧号对齐图片和动作，不能依赖查询返回的行顺序。
    by_frame = {r["frame_index"]: r for r in rows}
    if len(rows) != len(by_frame) or set(by_frame) != set(range(start, end)):
        raise ValueError("Prepared 窗口缺帧或重复，与样本索引不一致")
    action_frames = list(range(index["action_start"], index["action_end_exclusive"]))
    observation_frames = range(index["observation_start"], index["observation_end_exclusive"])
    # 若 Prepared 的动作或状态已改变，停止展示，避免配错画面。
    np.testing.assert_allclose([by_frame[f]["action"] for f in action_frames], targets[i], rtol=1e-5, atol=1e-6)
    np.testing.assert_allclose([by_frame[f]["observation_state"] for f in observation_frames], states[i], rtol=1e-5, atol=1e-6)
    return by_frame


def action_chart(actual, prediction, frames, cursor):
    # SVG 直接在 Notebook 显示，无需 matplotlib 或单独的 Web 服务。
    low = float(min(np.min(actual), np.min(prediction)))
    high = float(max(np.max(actual), np.max(prediction)))
    padding = max((high - low) * 0.1, 1e-5)
    low, high = low - padding, high + padding
    def x(j): return 65 + 630 * j / max(len(frames) - 1, 1)
    def y(v): return 230 - 190 * (float(v) - low) / (high - low)
    parts = ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 285" style="width:100%;max-width:850px;background:white">']
    for j in range(5):
        v = low + (high - low) * j / 4
        parts.append(f'<line x1="65" y1="{y(v)}" x2="695" y2="{y(v)}" stroke="#ddd"/><text x="3" y="{y(v)+4}" font-size="11">{v:.4g}</text>')
    for values, color, dash in [(actual, "#1677ff", ""), (prediction, "#e65c00", 'stroke-dasharray="6 4"')]:
        points = " ".join(f"{x(j)},{y(v)}" for j,v in enumerate(values))
        parts.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2" {dash}/>')
        for j,v in enumerate(values):
            parts.append(f'<circle cx="{x(j)}" cy="{y(v)}" r="2" fill="{color}"/>')
    if cursor is not None:
        parts.append(f'<line x1="{x(cursor)}" y1="35" x2="{x(cursor)}" y2="235" stroke="#777" stroke-dasharray="3 3"/>')
    for j in sorted({0, len(frames)//2, len(frames)-1}):
        parts.append(f'<text x="{x(j)}" y="250" text-anchor="middle" font-size="12">{frames[j]}</text>')
    parts.append('<text x="65" y="20" fill="#1677ff">真实 action</text><text x="240" y="20" fill="#e65c00">预测 action</text><text x="280" y="275">原始 frame_index（动作窗口）</text></svg>')
    return "".join(parts)


## 第 4 步：选择验证样本并加载

- 选择样本后点击加载；切换动作维度无需重新查库。
- 图片范围可选“观察窗口”或“动作目标窗口”。默认展示动作目标窗口，图片是真实采集时的画面，便于定位预测误差。
- 同一个样本的预测是一次完整动作窗口预测；拖动帧滑块只移动画面和曲线游标，不重新预测，也不是每帧重新控制机器人。
- 观察窗口中不属于动作目标范围的帧不会画预测游标。一个窗口通常很短，不能当成完整 episode 的执行演示。

选择验证样本 → 点击加载 → 播放或拖动滑块。左侧是采集画面，右侧是所选动作维度的对比曲线。这里的播放器代码只负责按钮、滑块和刷新显示，不参与训练。播放速度用于检查画面，不保证按采集时间实时播放。


In [ ]:
if globals().get("prediction_link") is not None:
    prediction_link.unlink()
if globals().get("prediction_play") is not None:
    prediction_play.playing = False
if globals().get("prediction_viewer") is not None:
    prediction_viewer.close()

sample_choice = widgets.Dropdown(options=[(f"{i}: {r['episode_id']} / anchor {r['anchor_frame']}", i) for i,r in enumerate(indices)], description="验证样本", layout=widgets.Layout(width="65%"))
load_prediction = widgets.Button(description="加载样本")
dimension = widgets.Dropdown(options=[(f"action[{d}]", d) for d in range(targets.shape[-1])], description="动作维度")
frame_mode = widgets.Dropdown(options=[("动作目标窗口", "action"), ("观察窗口", "observation")], description="图片范围")
prediction_play = widgets.Play(value=0, min=0, max=0, interval=150, disabled=True)
prediction_slider = widgets.IntSlider(value=0, min=0, max=0, description="窗口位置", continuous_update=False, disabled=True)
prediction_link = widgets.jslink((prediction_play,"value"),(prediction_slider,"value"))
prediction_info = widgets.HTML(value="选择样本并点击加载。")
prediction_images, prediction_panels, prediction_grid = make_camera_grid(width=320)
prediction_plot = widgets.HTML(layout=widgets.Layout(width="740px"))
loaded_sample = None
loaded_rows = {}
display_frames = []


# 只刷新当前帧、当前动作维度；拖动滑块不会重新预测或查询数据库。
def render_prediction(change=None):
    if loaded_sample is None or not display_frames:
        return
    i, d = loaded_sample, dimension.value
    index = indices[i]
    f = display_frames[prediction_slider.value]
    row = loaded_rows[f]
    timestamp = row.get("timestamp")
    time_text = "未知" if timestamp is None else f"{timestamp:.4f}s"
    actual, pred = targets[i,:,d], predicted[i,:,d]
    frames = list(range(index["action_start"], index["action_end_exclusive"]))
    cursor = frames.index(f) if f in frames else None
    chart = action_chart(actual, pred, frames, cursor)
    value_text = (f"真实 {actual[cursor]:.5f}；预测 {pred[cursor]:.5f}；绝对误差 {abs(actual[cursor]-pred[cursor]):.5f}"
                  if cursor is not None else "当前观察帧不在动作预测窗口内")
    prediction_info.value = (
        f"<b>真实采集画面（不是模型执行）</b> | {html.escape(index['episode_id'])} | val | anchor {index['anchor_frame']}<br>"
        f"frame_index {f} | timestamp {time_text} | action[{d}]：{value_text}<br>"
        f"本窗口该维度 MAE：{np.mean(np.abs(actual-pred)):.5f}（原始数值尺度）")
    update_camera_images(prediction_images, prediction_panels, row)
    prediction_plot.value = chart


def change_frame_mode(change=None):
    global display_frames
    prediction_play.playing = False
    display_frames = []
    prediction_play.value = prediction_slider.value = 0
    if loaded_sample is None:
        return
    index = indices[loaded_sample]
    key = frame_mode.value
    display_frames = list(range(index[f"{key}_start"], index[f"{key}_end_exclusive"]))
    prediction_play.max = prediction_slider.max = len(display_frames)-1
    prediction_play.disabled = prediction_slider.disabled = False
    render_prediction()


# 切换样本或加载失败时清空旧结果，避免误以为旧画面属于新样本。
def clear_prediction(change=None):
    global loaded_sample, loaded_rows, display_frames
    prediction_play.playing = False
    prediction_play.disabled = prediction_slider.disabled = True
    loaded_sample, loaded_rows, display_frames = None, {}, []
    update_camera_images(prediction_images, prediction_panels)
    prediction_plot.value = ""
    prediction_info.value = "选择样本后点击加载。"


# 点击加载才回读数据；之后播放直接使用内存中的这个窗口。
def load_prediction_sample(_):
    global loaded_sample, loaded_rows
    clear_prediction()
    prediction_info.value = "正在回读 Prepared 并核对 state/action…"
    try:
        i = sample_choice.value
        loaded_rows = read_window(i)
        loaded_sample = i
        change_frame_mode()
    except Exception as error:
        clear_prediction()
        prediction_info.value = "加载失败：" + html.escape(str(error))


sample_choice.observe(clear_prediction, names="value")
load_prediction.on_click(load_prediction_sample)
dimension.observe(render_prediction, names="value")
frame_mode.observe(change_frame_mode, names="value")
prediction_slider.observe(render_prediction, names="value")
prediction_viewer = widgets.VBox([
    widgets.HBox([sample_choice, load_prediction]),
    widgets.HBox([dimension, frame_mode]),
    widgets.HBox([prediction_play, prediction_slider]), prediction_info,
    widgets.HBox([
        prediction_grid,
        prediction_plot,
    ], layout=widgets.Layout(flex_flow="row wrap")),
])
display(prediction_viewer)
